# LinkML Notebook

The idea of this notebook is to create an environemnt that allows to rapidly iterate over different LinkML schemas and data files to check the different features of LinkML and how they work in practice. The notebook is not meant to be a tutorial on how to use LinkML, but rather a playground for testing different features and seeing how they work in practice.

The model and the data are defined in two seperate cells each with a yaml magic command that saves the content of the cell as python dictionary.

In [7]:
# register the yaml magic command

%reload_ext yamlmagic

## Helper Code

In [31]:
from linkml_runtime.linkml_model import SchemaDefinition
from linkml_runtime.utils.schemaview import SchemaView
from linkml_runtime.loaders import YAMLLoader
from linkml_runtime.dumpers import RDFLibDumper
from linkml_runtime.dumpers import JSONDumper
from linkml.generators.pythongen import PythonGenerator

import types

def process_linkml(schema, data, target_class_name):
    
    # load schema and create a schemaview
    schema = YAMLLoader().load(schema, target_class=SchemaDefinition)
    view = SchemaView(schema)

    # make real python classes from the schema
    python_code = PythonGenerator(schema).serialize()

    # create a dynamic module and execute the generated code in it
    module = types.ModuleType("my_schema")
    exec(python_code, module.__dict__)

    # load the data into an instance of a class defined in the schema
    data = YAMLLoader().load(data, target_class=getattr(module, target_class_name))

    # convert the data to RDF and print it
    print("RDF:\n")
    print(RDFLibDumper().dumps(data, schemaview=view))

    # convert the data to JSON and print it
    print("\nJSON:\n")
    print(JSONDumper().dumps(data))
    

## Basic Model

see: https://linkml.io/linkml/intro/tutorial01.html

In [22]:
%%yaml schema

id: https://w3id.org/linkml/examples/personinfo
name: personinfo
prefixes:
  linkml: https://w3id.org/linkml/
  personinfo: https://w3id.org/linkml/examples/personinfo
imports:
  - linkml:types
default_range: string
default_prefix: personinfo

classes:
  Person:
    attributes:
      id:
      full_name:
      aliases:
      phone:
      age:

<IPython.core.display.Javascript object>

In [23]:
%%yaml data

id: ORCID:1234
full_name: Clark Kent
age: "32"
phone: 555-555-5555

<IPython.core.display.Javascript object>

In [32]:
process_linkml(schema, data, "Person")

RDF:

@prefix personinfo: <https://w3id.org/linkml/examples/personinfo> .

[] a personinfo:Person ;
    personinfo:age "32" ;
    personinfo:full_name "Clark Kent" ;
    personinfo:id "ORCID:1234" ;
    personinfo:phone "555-555-5555" .



JSON:

{
  "id": "ORCID:1234",
  "full_name": "Clark Kent",
  "phone": "555-555-5555",
  "age": "32",
  "@type": "Person"
}


## References

In [33]:
%%yaml schema

id: https://ld.ech.ch/tutorial/schema_references
name: schema_references
description: |
  [en] This is a dummy schema to demonstrate the use of references to other schemas in LinkML. It imports the `schema_dates` schema to reuse the date and time attributes defined there
prefixes:
  xsd: http://www.w3.org/2001/XMLSchema#
  prov: http://www.w3.org/ns/prov#
  dcterms: http://purl.org/dc/terms/
  linkml: https://w3id.org/linkml/
  schema: http://schema.org/
  tut: https://ld.ech.ch/tutorial/schema_references/
imports:
  - linkml:types
  - input/schema_dates
default_prefix: tut
default_range: string
  
classes:
  
  Reference:
    description: A reference to another entity, e.g. a person, organization, or location. This class can be used as a base class for more specific reference types. This class is not meant to be instantiated directly, but serves as a base for other reference classes.
    abstract: true
    slots:
      - id
      - datetime_created

  PersonReference:
    is_a: Reference
    description: A reference to a person, e.g. a politician, a member of parliament, or a government official.
    slots:
      - name
      - party
      - position

  Container:
    description: A container for references.
    tree_root: true
    slots:
      - id
      - person_references

slots:

  id:
    identifier: true
    required: true

  name:
    slot_uri: schema:name

  party:
    slot_uri: tut:party

  position:
    slot_uri: tut:position

  person_references:
    slot_uri: tut:personReference
    range: PersonReference
    multivalued: true
    inlined_as_list: true


<IPython.core.display.Javascript object>

In [34]:
%%yaml data

id: tut:references-1
person_references:
  - id: tut:p1
    name: John Doe
    party: Independent
    position: Member of Parliament
  - id: tut:p2
    name: Jane Smith
    party: Green Party
    position: Minister of Environment

<IPython.core.display.Javascript object>

In [35]:
process_linkml(schema, data, "Container")

RDF:

@prefix schema1: <http://schema.org/> .
@prefix tut: <https://ld.ech.ch/tutorial/schema_references/> .

tut:references-1 a tut:Container ;
    tut:personReference tut:p1,
        tut:p2 .

tut:p1 a tut:PersonReference ;
    schema1:name "John Doe" ;
    tut:party "Independent" ;
    tut:position "Member of Parliament" .

tut:p2 a tut:PersonReference ;
    schema1:name "Jane Smith" ;
    tut:party "Green Party" ;
    tut:position "Minister of Environment" .



JSON:

{
  "id": "tut:references-1",
  "person_references": [
    {
      "id": "tut:p1",
      "name": "John Doe",
      "party": "Independent",
      "position": "Member of Parliament"
    },
    {
      "id": "tut:p2",
      "name": "Jane Smith",
      "party": "Green Party",
      "position": "Minister of Environment"
    }
  ],
  "@type": "Container"
}
